# Project: A BPE tokenizer trained on my notes 🔤
GPT models don't read letters or words: they read **tokens** made by byte-pair encoding (BPE). Start from characters and repeatedly merge the most frequent adjacent pair. Train BPE on your Danish IT-ret notes and watch it discover *data*, *behandling*, *registrerede*, then measure how many tokens a sentence costs (which is what you pay for in an LLM API).

Topic: [[LLMs Overview]], [[Text Representations]] · guide note: [[Project - BPE Tokenizer on My Notes]] · paper: [Sennrich et al. 2016](https://arxiv.org/abs/1508.07909)

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import collections, pathlib, re
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
text = " ".join(f.read_text(encoding="utf-8").lower() for f in pathlib.Path("../../9-semester/IT-lov").rglob("*.md"))
words = re.findall(r"[a-zæøå]+", text)
vocab0 = collections.Counter(tuple(w) + ("</w>",) for w in words)       # words as character tuples; </w> marks the end
print(f"{len(words):,} words, {len(vocab0):,} distinct; top: {collections.Counter(words).most_common(8)}")

## 1. Count adjacent pairs
`vocab` maps a word (tuple of symbols) to its frequency. Return a dict `{(a, b): count}` over all adjacent symbol pairs, weighted by word frequency.

In [ ]:
def pair_counts(vocab):
    # TODO 1: frequency-weighted counts of adjacent symbol pairs
    raise NotImplementedError  # TODO 1

## 2. Merge a pair
Return a new vocab where every occurrence of `pair = (a, b)` inside a word becomes the single symbol `a + b` (scan left to right, non-overlapping).

In [ ]:
def merge(vocab, pair):
    # TODO 2: replace adjacent (a, b) by "ab" in every word
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_bpe():
    v = {("a", "a", "a", "</w>"): 2, ("b", "a", "</w>"): 1}
    pc = pair_counts(v); m = merge(v, ("a", "a"))
    return pc[("a", "a")] == 4 and pc[("b", "a")] == 1 and m == {("aa", "a", "</w>"): 2, ("b", "a", "</w>"): 1}
check("BPE steps", _t_bpe, "'aaa' contributes (a,a) twice ×2 = 4; merging gives ('aa','a') (non-overlapping, left to right).")

def train_bpe(vocab, n_merges):
    merges = []
    for _ in range(n_merges):
        pc = pair_counts(vocab)
        if not pc: break
        best = max(pc, key=pc.get); vocab = merge(vocab, best); merges.append(best)
    return merges

def encode(word, merges):
    sym = tuple(word) + ("</w>",)
    for p in merges: sym = next(iter(merge({sym: 1}, p)))
    return list(sym)

if ready("BPE steps"):
    merges = train_bpe(vocab0, 500)
    print("first 15 merges:", ["".join(m) for m in merges[:15]])
    tokens = {"".join(m) for m in merges}
    print("learned whole words:", sorted(t[:-4] for t in tokens if t.endswith("</w>") and len(t) > 9)[:25])
    sent = "den dataansvarlige skal kunne påvise at behandlingen af personoplysninger er lovlig"
    toks = [t for w in sent.split() for t in encode(w, merges)]
    print(f"\n{len(sent.split())} words → {len(toks)} BPE tokens vs {len(sent.replace(' ', ''))} characters"); print(toks)
    check("compresses Danish legalese", lambda: len(toks) < 0.35 * len(sent.replace(" ", "")),
          "After 500 merges a sentence from your notes should need < 35 % as many tokens as characters.")

<details><summary>▶ Solution</summary>

```python
def pair_counts(vocab):
    counts = {}
    for word, freq in vocab.items():
        for a, b_ in zip(word, word[1:]):
            counts[(a, b_)] = counts.get((a, b_), 0) + freq
    return counts
```

```python
def merge(vocab, pair):
    new = {}
    for word, freq in vocab.items():
        out, i = [], 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                out.append(word[i] + word[i + 1]); i += 2
            else:
                out.append(word[i]); i += 1
        new[tuple(out)] = new.get(tuple(out), 0) + freq
    return new
```
</details>

## Stretch goals
- Encode an **English** ML sentence with the Danish BPE. Why does it explode into many tokens? (This is why LLMs cost more in some languages.)
- Plot tokens per word vs. number of merges (50 … 2000).
- Use the BPE tokens instead of characters in [[Mini-GPT on Danish]].